In [2]:
import torch
import torch.nn as nn
import joblib
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset


class AutoEncoder(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 16),
            nn.ReLU(),
            nn.Linear(16, 8)
        )
        self.decoder = nn.Sequential(
            nn.Linear(8, 16),
            nn.ReLU(),
            nn.Linear(16, input_dim)
        )

    def forward(self, x):
        return self.decoder(self.encoder(x))


def train_autoencoder(csv_path, model_save_path, scaler_save_path, epochs=50, batch_size=256, lr=1e-3):
    df = pd.read_csv(csv_path)

    # ✅ is_fraud 제외한 수치형 컬럼만 사용
    feature_cols = [c for c in df.select_dtypes(include=[np.number]).columns if c != "is_fraud"]
    print(f"학습 feature 수: {len(feature_cols)}개 → {feature_cols}")

    X = df[feature_cols].values

    # scaler fit & 저장
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    joblib.dump(scaler, scaler_save_path)
    print(f"✅ scaler 저장 완료: {scaler_save_path}")

    # 모델 학습
    X_tensor = torch.FloatTensor(X_scaled)
    loader = DataLoader(TensorDataset(X_tensor), batch_size=batch_size, shuffle=True)

    model = AutoEncoder(input_dim=len(feature_cols))
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()

    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for (batch,) in loader:
            optimizer.zero_grad()
            output = model(batch)
            loss = criterion(output, batch)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        if (epoch + 1) % 10 == 0:
            print(f"Epoch {epoch+1}/{epochs} | Loss: {total_loss/len(loader):.6f}")

    torch.save(model.state_dict(), model_save_path)
    print(f"✅ 모델 저장 완료: {model_save_path}")



In [ ]:

def calculate_anomaly_score(csv_path, model_path, scaler_path, output_path):
    df = pd.read_csv(csv_path)

    # ✅ is_fraud 제외
    feature_cols = [c for c in df.select_dtypes(include=[np.number]).columns if c != "is_fraud"]

    state_dict = torch.load(model_path, weights_only=True)
    input_dim = state_dict["encoder.0.weight"].shape[1]

    model = AutoEncoder(input_dim=input_dim)
    model.load_state_dict(state_dict)
    model.eval()

    scaler = joblib.load(scaler_path)

    X = df[feature_cols].values
    X_scaled = scaler.transform(X)
    X_tensor = torch.FloatTensor(X_scaled)

    with torch.no_grad():
        recon = model(X_tensor)

    loss = ((X_tensor - recon) ** 2).mean(dim=1).numpy()

    denom = loss.max() - loss.min()
    anomaly_score = np.zeros_like(loss) if denom == 0 else (loss - loss.min()) / denom * 100

    df = df.copy()
    df["anomaly_score"] = anomaly_score.round(3)
    df.to_csv(output_path, index=False)
    print(f"✅ anomaly_score 생성 완료: {output_path}")

    return df


if __name__ == "__main__":
    CSV_PATH    = "../data/cc_fraud_train_processed.csv"
    MODEL_PATH  = "../model/autoencoder_anomaly.pth"
    SCALER_PATH = "../model/scaler_anomaly.pkl"
    OUTPUT_PATH = "../data/anomaly_scored.csv"

    # 1단계: 재학습 (is_fraud 제외)
    train_autoencoder(CSV_PATH, MODEL_PATH, SCALER_PATH)

    # 2단계: anomaly score 계산
    df = calculate_anomaly_score(CSV_PATH, MODEL_PATH, SCALER_PATH, OUTPUT_PATH)

학습 feature 수: 17개 → ['amt', 'zip', 'lat', 'long', 'city_pop', 'unix_time', 'merch_lat', 'merch_long', 'trans_hour', 'trans_dayofweek', 'trans_month', 'trans_day', 'age', 'distance_km', 'amt_zscore', 'hour_dev', 'high_amt_far']
✅ scaler 저장 완료: ../model/scaler_anomaly.pkl
Epoch 10/50 | Loss: 0.091443
Epoch 20/50 | Loss: 0.088990
